In [2]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.aih_privacy.datasets.sisfall as sis
importlib.reload(sis)

# Project Imports
from src.aih_privacy.datasets.registry import get_dataset
from src.aih_privacy.datasets.sisfall.load import iter_files, load_file, parse_filename
from src.aih_privacy.datasets.sisfall.preprocess import rename_axes, apply_filter_by_trial, apply_filter_by_trial_v2
from src.aih_privacy.features.sisfall_features import extract_window_features_one_df, extract_comprehensive_features_one_df_v2
from src.aih_privacy.config import DATA_PROCESSED_DIR 

In [4]:
FS = 200.0

WIN_SEC = 1.0
STEP_SEC = 0.5

WIN_SIZE = int(FS * WIN_SEC)
STEP = int(FS * STEP_SEC)

FILTER_CUTOFF = 5.0
FILTER_ORDER = 4

MAX_FILES = 1000

tag = "win1.00_step0.50_f5.0"

dataset = get_dataset("sisfall")
files = list(iter_files(dataset.raw_dir))

In [ ]:


if MAX_FILES is not None:
    files = files[:MAX_FILES]

rows = []

print(f"Starting processing of {len(files)} files...")

# Using tqdm for a progress bar
for f in tqdm(files, desc="Processing Trials"):
    try:
        # 1. Parse Metadata
        m = parse_filename(f)
        if m is None:
            continue

        trial_id = m["trial_id"]
        label = int(m["label"])
        subject_id = m["subject_id"]
        age_group = m["age_group"]
        activity_code = m["activity_code"]

        # 2. Load Data
        df = load_file(f)
        df = rename_axes(df) # Standardize to ax, ay, az, gx, gy, gz

        # 3. Preprocessing (Add index and Filter)
        df = df.reset_index(drop=True)
        df["sample_idx"] = np.arange(len(df), dtype=np.int32)
        df["trial_id"] = trial_id

        # Apply Low Pass Filter (Butterworth)
        df = apply_filter_by_trial(
            df,
            cutoff_hz=FILTER_CUTOFF,
            fs_hz=FS,
            order=FILTER_ORDER,
            cols=("ax", "ay", "az"), # Usually applied to Accel
        )

        # 4. Feature Extraction (C1, C2, C8, C9, etc.)
        # This function must exist in your src.aih_privacy.features.sisfall_features
        wf = extract_window_features_one_df(df, FS, WIN_SEC, STEP_SEC)

        if wf.empty:
            continue

        # 5. Attach Labels/Metadata to the Windowed Data
        wf["trial_id"] = trial_id
        wf["label"] = label
        wf["subject_id"] = subject_id
        wf["age_group"] = age_group
        wf["activity_code"] = activity_code

        rows.append(wf)

    except Exception as e:
        print(f"Error processing {f.name}: {e}")
        continue

# Concatenate all results
if rows:
    windows_df = pd.concat(rows, ignore_index=True)
    print("Processing complete.")
else:
    print("No data processed.")
    windows_df = pd.DataFrame()

Starting processing of 4505 files...


Processing Trials:   0%|          | 0/4505 [00:00<?, ?it/s]

Processing complete.


In [5]:
out_dir = DATA_PROCESSED_DIR / "sisfall"

# 1. FILTERING: Select only ADL (Activities of Daily Living)
# We exclude Falls (F01-F15) because they don't have a consistent "gait".
# D01-D09 are walking/jogging/stairs activities best for ID.
adl_activities = [
    'D01', 'D02', 'D03', 'D04', 'D05', 
    'D06', 'D07', 'D08', 'D09', 'D10', 
    'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17'
]

# Create the ID dataset
df_id = windows_df[windows_df['activity_code'].isin(adl_activities)].copy()

# 2. FEATURE CHECK
# Ensure you are using the per-window features (c2_max, c8, c9, c1_max)
# NOT the aggregated ones (c2_max_max).
print(f"Original shape: {windows_df.shape}")
print(f"Identification shape (ADL only): {df_id.shape}")

# 3. SAVE
# Save this specifically for the Identification task
out_id_path = out_dir / f"windows_for_identification_{tag}.parquet"
df_id.to_parquet(out_id_path, index=False)
print(f"Saved to: {out_id_path}")

Original shape: (153705, 9)
Identification shape (ADL only): (96174, 9)
Saved to: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\windows_for_identification_win1.00_step0.50_f5.0.parquet


In [ ]:
if not windows_df.empty:
    # 1. Inspect Data
    print(f"Shape: {windows_df.shape}")
    print(f"Label Distribution:\n{windows_df['label'].value_counts()}")
    print(f"Unique Subjects: {windows_df['subject_id'].nunique()}")
    
    # 2. Define Output Path
    # out_dir = DATA_PROCESSED_DIR / "sisfall"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Naming convention includes parameters for reproducibility
    tag = f"win{WIN_SEC:.2f}_step{STEP_SEC:.2f}_f{FILTER_CUTOFF:.1f}"
    out_path = out_dir / f"windows_df_{tag}.parquet"

    # 3. Save
    windows_df.to_parquet(out_path, index=False)
    print(f"Saved successfully to: {out_path}")
    
    # Display preview
    display(windows_df.head())
else:
    print("DataFrame is empty, nothing to save.")

Shape: (153705, 9)
Label Distribution:
label
0    101686
1     52019
Name: count, dtype: int64
Unique Subjects: 38
Saved successfully to: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\windows_df_win1.00_step0.50_f5.0.parquet


,c2_max,c8,c9,c1_max,trial_id,label,subject_id,age_group,activity_code
0,0.308215,0.144923,0.326935,1.338845,D01_SA01_R01,0,SA01,SA,D01
1,0.397239,0.200530,0.237491,1.338845,D01_SA01_R01,0,SA01,SA,D01
2,0.400025,0.225428,0.256905,1.321054,D01_SA01_R01,0,SA01,SA,D01
3,0.400025,0.178712,0.227607,1.321054,D01_SA01_R01,0,SA01,SA,D01
4,0.371433,0.165369,0.204123,1.302363,D01_SA01_R01,0,SA01,SA,D01


In [5]:
GROUP_COLS = ["trial_id", "label", "subject_id", "age_group", "activity_code"]

df_trials = (
    windows_df
    .groupby(GROUP_COLS, as_index=False)
    .agg(
        n_windows=("c2_max", "count"),
        c2_max_max=("c2_max", "max"),
        c2_max_mean=("c2_max", "mean"),
        c2_max_std=("c2_max", "std"),
        c8_mean=("c8", "mean"),
        c8_max=("c8", "max"),
        c9_mean=("c9", "mean"),
        c1_max_max=("c1_max", "max"),
    )
)

df_trials.head()

,trial_id,label,subject_id,age_group,activity_code,n_windows,c2_max_max,c2_max_mean,c2_max_std,c8_mean,c8_max,c9_mean,c1_max_max
0,D01_SA01_R01,0,SA01,SA,D01,198,0.563236,0.418996,0.037546,0.185930,0.231747,0.237608,1.513698
1,D01_SA02_R01,0,SA02,SA,D01,198,0.471547,0.360264,0.040570,0.188866,0.236531,0.261376,1.535287
2,D01_SA03_R01,0,SA03,SA,D01,198,0.753453,0.671717,0.062895,0.190047,0.241245,0.243012,1.474569
3,D01_SA04_R01,0,SA04,SA,D01,198,0.751347,0.657644,0.056495,0.170927,0.233457,0.218769,1.493332
4,D01_SA05_R01,0,SA05,SA,D01,198,0.445957,0.358605,0.042250,0.179623,0.236440,0.223999,1.454516


In [6]:
trials_path = out_dir / f"df_trials_{tag}.parquet"
df_trials.to_parquet(trials_path, index=False)
print("Saved:", trials_path)

Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\df_trials_win1.00_step0.50_f5.0.parquet


In [ ]:
FILTER_CUTOFF = 20.0 
WIN_SEC = 3.0     
STEP_SEC = 1.5

ADL_ACTIVITIES = [
    'D01', 'D02', 'D03', 'D04', 'D05', 
    'D06', 'D07', 'D08', 'D09', 'D10', 
    'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17'
]

# ADL_ACTIVITIES = ['D01', 'D02', 'D03', 'D04', 'D08', 'D09'] 


In [ ]:
dataset = get_dataset("sisfall")
files = list(iter_files(dataset.raw_dir))
    
rows = []

print(f"Starting processing of {len(files)} files...")

for f in tqdm(files, desc="Processing Trials for Subject ID"):
    try:
        # 1. Parse Metadata
        m = parse_filename(f)
        if m is None:
            continue

        trial_id = m["trial_id"]
        label = int(m["label"])
        subject_id = m["subject_id"]
        age_group = m["age_group"]
        activity_code = m["activity_code"]

        # OPTIMIZATION 2: Filter Data Selection
        # Skip Falls immediately. They confuse the Identity model.
        if activity_code not in ADL_ACTIVITIES:
            continue

        # 2. Load Data
        df = load_file(f)
        df = rename_axes(df) # Standardize to ax, ay, az, gx, gy, gz

        # 3. Preprocessing
        df = df.reset_index(drop=True)
        df["sample_idx"] = np.arange(len(df), dtype=np.int32)
        df["trial_id"] = trial_id

        # OPTIMIZATION 3: Filter ALL axes (including Gyroscope)
        # Gyroscope data (rotation) is crucial for identifying hip movement style.
        # Ensure your apply_filter_by_trial_v2 handles missing cols gracefully.
        df = apply_filter_by_trial_v2(
            df,
            cutoff_hz=FILTER_CUTOFF, # ~20Hz
            fs_hz=FS,
            order=FILTER_ORDER,
            cols=("ax", "ay", "az", "gx", "gy", "gz"), 
        )

        # 4. Feature Extraction
        # USE THE NEW FUNCTION: extract_comprehensive_features_one_df
        # This extracts Mean, Std, Skew, Kurtosis for every axis.
        # If you use the old one (only C1, C2...), accuracy will stay low.
        wf = extract_comprehensive_features_one_df_v2(df, FS, WIN_SEC, STEP_SEC)

        if wf.empty:
            continue

        # 5. Attach Metadata
        wf["trial_id"] = trial_id
        # For Subject ID, 'label' is usually the subject, but we keep metadata to split later
        wf["label"] = label 
        wf["subject_id"] = subject_id
        wf["age_group"] = age_group
        wf["activity_code"] = activity_code

        rows.append(wf)

    except Exception as e:
        print(f"Error processing {f.name}: {e}")
        continue

Starting processing of 4505 files...


Processing Trials for Subject ID:   0%|          | 0/4505 [00:00<?, ?it/s]

In [15]:
if rows:
    wf = pd.concat(rows, ignore_index=True)
    # Save specifically for ID so you don't mix it with your Fall Detection work
    out_path = DATA_PROCESSED_DIR / "sisfall" / f"windows_ID_comprehensive_{tag}.parquet"
    wf.to_parquet(out_path, index=False)
    print(f"Saved {len(wf)} windows for Identification.")

Saved 29951 windows for Identification.


In [16]:
print(f"Features extraídas (exemplo): {wf.columns}...")


Features extraídas (exemplo): Index(['ax_mean', 'ax_std', 'ax_max', 'ax_min', 'ax_skew', 'ax_kurt',
       'ay_mean', 'ay_std', 'ay_max', 'ay_min', 'ay_skew', 'ay_kurt',
       'az_mean', 'az_std', 'az_max', 'az_min', 'az_skew', 'az_kurt',
       'gx_mean', 'gx_std', 'gx_max', 'gx_min', 'gx_skew', 'gx_kurt',
       'gy_mean', 'gy_std', 'gy_max', 'gy_min', 'gy_skew', 'gy_kurt',
       'gz_mean', 'gz_std', 'gz_max', 'gz_min', 'gz_skew', 'gz_kurt',
       'mag_mean', 'mag_std', 'trial_id', 'label', 'subject_id', 'age_group',
       'activity_code'],
      dtype='object')...


In [17]:
for c in wf.columns:
    print(c)

ax_mean
ax_std
ax_max
ax_min
ax_skew
ax_kurt
ay_mean
ay_std
ay_max
ay_min
ay_skew
ay_kurt
az_mean
az_std
az_max
az_min
az_skew
az_kurt
gx_mean
gx_std
gx_max
gx_min
gx_skew
gx_kurt
gy_mean
gy_std
gy_max
gy_min
gy_skew
gy_kurt
gz_mean
gz_std
gz_max
gz_min
gz_skew
gz_kurt
mag_mean
mag_std
trial_id
label
subject_id
age_group
activity_code


In [6]:
from scipy.signal import find_peaks
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, iqr
from src.aih_privacy.features.windowing import sliding_windows_indices

def extract_hybrid_features_one_df(df, fs=200.0, win_sec=3.0, step_sec=1.5):
    """
    V3: Inclui Estatísticas (Tempo), Magnitudes (Invariante a Rotação) 
    e Frequência (FFT - Ritmo).
    """
    win = int(fs * win_sec)
    step = int(fs * step_sec)
    
    features_list = []
    
    # Pré-calcular Magnitudes (Vetor Resultante) - Crucial para invariância de rotação
    # Se o sensor girar, ax muda, mas acc_mag continua igual.
    df['acc_mag'] = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2)
    df['gyro_mag'] = np.sqrt(df['gx']**2 + df['gy']**2 + df['gz']**2)
    
    # Lista de sinais para extrair
    signals = ['ax', 'ay', 'az', 'gx', 'gy', 'gz', 'acc_mag', 'gyro_mag']
    
    for s, e in sliding_windows_indices(len(df), win, step):
        window = df.iloc[s:e]
        
        if len(window) < win: continue
        
        # Filtro de Silêncio: Se a variância da magnitude for muito baixa, a pessoa está parada.
        if window['acc_mag'].std() < 0.02: 
            continue

        row = {}
        
        # 1. Domínio do Tempo (Estatísticas)
        for col in signals:
            arr = window[col].values
            row[f'{col}_mean'] = np.mean(arr)
            row[f'{col}_std']  = np.std(arr)
            row[f'{col}_max']  = np.max(arr)
            row[f'{col}_min']  = np.min(arr)
            row[f'{col}_iqr']  = iqr(arr) # Intervalo Interquartil (Robusto a outliers)
            row[f'{col}_skew'] = skew(arr)
            row[f'{col}_kurt'] = kurtosis(arr)
            # Zero Crossing Rate (bom para frequência estimada no tempo)
            row[f'{col}_zcr'] = ((arr[:-1] * arr[1:]) < 0).sum()

        # 2. Domínio da Frequência (FFT) - O "Ritmo" da pessoa
        # Apenas na Magnitude para economizar computação e ser robusto a rotação
        for col in ['acc_mag', 'gyro_mag']:
            arr = window[col].values
            # Remover componente DC (gravidade média) para focar na oscilação
            arr_no_dc = arr - np.mean(arr)
            
            # FFT
            fft_vals = np.abs(np.fft.rfft(arr_no_dc))
            fft_freqs = np.fft.rfftfreq(len(arr_no_dc), d=1/fs)
            
            # Dominant Frequency (Cadência da passada)
            dom_idx = np.argmax(fft_vals)
            row[f'{col}_dom_freq'] = fft_freqs[dom_idx]
            
            # Energy (Força da passada)
            row[f'{col}_spec_energy'] = np.sum(fft_vals**2) / len(fft_vals)
            
            # Spectral Entropy (Complexidade do movimento)
            psd_norm = fft_vals / np.sum(fft_vals)
            # Evitar log(0)
            psd_norm = psd_norm[psd_norm > 0]
            row[f'{col}_spec_entropy'] = -np.sum(psd_norm * np.log(psd_norm))

        # Metadados
        if 'subject_id' in df.columns:
            row['subject_id'] = df['subject_id'].iloc[0]
            row['trial_id'] = df['trial_id'].iloc[0]
            row['activity_code'] = df['activity_code'].iloc[0]
            
        features_list.append(row)
        
    return pd.DataFrame(features_list)

In [7]:
rows_full = []

print("Processando TODOS os ficheiros (ADL + Falls) para teste de Utilidade...")

for f in tqdm(files, desc="Processing Full Dataset"):
    try:
        m = parse_filename(f)
        if m is None: continue

        # AQUI: Não filtramos por TARGET_ACTIVITIES. Queremos Dxx e Fxx.
        
        df = load_file(f)
        df = rename_axes(df)

        df = df.reset_index(drop=True)
        df["sample_idx"] = np.arange(len(df), dtype=np.int32)
        df["trial_id"] = m["trial_id"]

        # Aplicar filtro em todos os eixos
        df = apply_filter_by_trial_v2(df, cutoff_hz=FILTER_CUTOFF, cols=("ax", "ay", "az", "gx", "gy", "gz"))

        # Extração de Features Híbridas
        wf = extract_hybrid_features_one_df(df, FS, WIN_SEC, STEP_SEC)

        if wf.empty: continue

        # Metadados importantes
        wf["subject_id"] = m["subject_id"]
        wf["trial_id"] = m["trial_id"]
        wf["activity_code"] = m["activity_code"]
        wf["label"] = int(m["label"]) # 0 para ADL, 1 para Queda

        rows_full.append(wf)

    except Exception as e:
        print(f"Erro em {f.name}: {e}")
        continue

# --- Salvar o Dataset de Utilidade Completo ---
if rows_full:
    df_util_full = pd.concat(rows_full, ignore_index=True)
    out_path = DATA_PROCESSED_DIR / "sisfall" / f"windows_FULL_hybrid_{tag}.parquet"
    df_util_full.to_parquet(out_path, index=False)
    print(f"Dataset completo salvo com {len(df_util_full)} janelas.")

Processando TODOS os ficheiros (ADL + Falls) para teste de Utilidade...


Processing Full Dataset:   0%|          | 0/4505 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [23]:
if rows:
    wf = pd.concat(rows, ignore_index=True)
    # Save specifically for ID so you don't mix it with your Fall Detection work
    out_path = DATA_PROCESSED_DIR / "sisfall" / f"windows_ID_comprehensive_hybrid_{tag}.parquet"
    wf.to_parquet(out_path, index=False)
    print(f"Saved {len(wf)} windows for Identification.")

Saved 25237 windows for Identification.


In [24]:
for c in wf.columns:
    print(c)

ax_mean
ax_std
ax_max
ax_min
ax_iqr
ax_skew
ax_kurt
ax_zcr
ay_mean
ay_std
ay_max
ay_min
ay_iqr
ay_skew
ay_kurt
ay_zcr
az_mean
az_std
az_max
az_min
az_iqr
az_skew
az_kurt
az_zcr
gx_mean
gx_std
gx_max
gx_min
gx_iqr
gx_skew
gx_kurt
gx_zcr
gy_mean
gy_std
gy_max
gy_min
gy_iqr
gy_skew
gy_kurt
gy_zcr
gz_mean
gz_std
gz_max
gz_min
gz_iqr
gz_skew
gz_kurt
gz_zcr
acc_mag_mean
acc_mag_std
acc_mag_max
acc_mag_min
acc_mag_iqr
acc_mag_skew
acc_mag_kurt
acc_mag_zcr
gyro_mag_mean
gyro_mag_std
gyro_mag_max
gyro_mag_min
gyro_mag_iqr
gyro_mag_skew
gyro_mag_kurt
gyro_mag_zcr
acc_mag_dom_freq
acc_mag_spec_energy
acc_mag_spec_entropy
gyro_mag_dom_freq
gyro_mag_spec_energy
gyro_mag_spec_entropy
trial_id
label
subject_id
age_group
activity_code


In [1]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Project paths
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project specific imports
from src.aih_privacy.datasets.registry import get_dataset
from src.aih_privacy.datasets.sisfall.load import iter_files, load_file, parse_filename
from src.aih_privacy.datasets.sisfall.preprocess import rename_axes, apply_filter_by_trial_v2
from src.aih_privacy.features.sisfall_features import extract_hybrid_features_one_df, extract_window_features_one_df
from src.aih_privacy.config import DATA_PROCESSED_DIR

In [2]:
dataset = get_dataset("sisfall")
files = list(iter_files(dataset.raw_dir))

In [3]:
FS = 200.0
FILTER_CUTOFF = 20.0
FILTER_ORDER = 4

# Parâmetros para o DATASET DE TRIALS (Deteção de Queda)
# Geralmente janelas curtas funcionam bem aqui porque depois agregamos no MAX
WIN_TRIAL = 1.0 
STEP_TRIAL = 0.5

# Parâmetros para o DATASET DE WINDOWS (Identificação)
# Janela mais longa (ex: 3s) para o modelo de janela conseguir "ver" o padrão de marcha
WIN_WINDOW = 3.0 
STEP_WINDOW = 1.5

tag_trial = f"win{WIN_TRIAL:.2f}_step{STEP_TRIAL:.2f}"
tag_window = f"win{WIN_WINDOW:.2f}_step{STEP_WINDOW:.2f}"

In [4]:
rows_for_trials = []
rows_for_windows = []

print("Processing files with dual-windowing strategy...")

for f in tqdm(files, desc="Processing"):
    try:
        m = parse_filename(f)
        if m is None: continue

        # 1. Carga e Filtro (Comum a ambos)
        df_raw = load_file(f)
        df_raw = rename_axes(df_raw)
        df_raw["sample_idx"] = np.arange(len(df_raw), dtype=np.int32)
        df_raw["trial_id"] = m["trial_id"]

        df_filtered = apply_filter_by_trial_v2(
            df_raw, cutoff_hz=FILTER_CUTOFF, fs_hz=FS, order=FILTER_ORDER,
            cols=("ax", "ay", "az", "gx", "gy", "gz")
        )

        # --- PROCESSAMENTO PARA TRIALS (Deteção de Queda) ---
        # Usamos os parâmetros de 1.0s e as features de magnitude originais
        wf_trial = extract_window_features_one_df(df_filtered, FS, WIN_TRIAL, STEP_TRIAL)
        if not wf_trial.empty:
            wf_trial["trial_id"] = m["trial_id"]
            wf_trial["subject_id"] = m["subject_id"]
            wf_trial["label"] = int(m["label"])
            wf_trial["activity_code"] = m["activity_code"]
            wf_trial["age_group"] = m["age_group"]
            rows_for_trials.append(wf_trial)

        # --- PROCESSAMENTO PARA WINDOWS (Identificação / Utilidade de Janela) ---
        # Usamos os parâmetros de 3.0s e as features híbridas ricas
        wf_win = extract_hybrid_features_one_df(df_filtered, FS, WIN_WINDOW, STEP_WINDOW)
        if not wf_win.empty:
            wf_win["trial_id"] = m["trial_id"]
            wf_win["subject_id"] = m["subject_id"]
            wf_win["label"] = int(m["label"])
            wf_win["activity_code"] = m["activity_code"]
            rows_for_windows.append(wf_win)

    except Exception as e:
        print(f"Error in {f.name}: {e}")
        continue

# Concatenar
df_all_windows = pd.concat(rows_for_windows, ignore_index=True)
df_all_trials_base = pd.concat(rows_for_trials, ignore_index=True)

Processing files with dual-windowing strategy...


Processing:   0%|          | 0/4505 [00:00<?, ?it/s]

In [5]:
out_dir = DATA_PROCESSED_DIR / "sisfall"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Dataset de Identificação (Janelas de 3s - Features Híbridas)
win_path = out_dir / f"df_windows_hybrid_{tag_window}.parquet"
df_all_windows.to_parquet(win_path, index=False)
print(f"Saved Hybrid Windows (3s): {win_path}")

# 2. Dataset de Deteção de Queda (Agregado por Trial - Features Originais)
# A agregação é feita sobre as janelas de 1s
GROUP_COLS = ["trial_id", "label", "subject_id", "age_group", "activity_code"]
df_trials_final = (
    df_all_trials_base
    .groupby(GROUP_COLS, as_index=False)
    .agg(
        n_windows=("c2_max", "count"),
        c2_max_max=("c2_max", "max"),
        c2_max_mean=("c2_max", "mean"),
        c2_max_std=("c2_max", "std"),
        c8_mean=("c8", "mean"),
        c8_max=("c8", "max"),
        c9_mean=("c9", "mean"),
        c1_max_max=("c1_max", "max"),
    )
)
trial_path = out_dir / f"df_trials_original_{tag_trial}.parquet"
df_trials_final.to_parquet(trial_path, index=False)
print(f"Saved Aggregated Trials (1s base): {trial_path}")

Saved Hybrid Windows (3s): C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\df_windows_hybrid_win3.00_step1.50.parquet
Saved Aggregated Trials (1s base): C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\df_trials_original_win1.00_step0.50.parquet
